# Coffee Data Exploration

Exploring the data extracted from coffeereviews.com

| Date | User | Change Type | Remarks |  
| ---- | ---- | ----------- | ------- |
| 17/03/2026   | Martin | Create  | Notebook created. Creating DuckDB from JSON | 
| 19/03/2026   | Martin | Update  | Updated "Agtron" and "Est. Price" fields due to misnaming. Exploration of document database. ChromaDB checks. | 

# Content

* [DuckDB Ingestion](#duckdb-ingestion)
* [Data Exploration](#data-exploration)

# DuckDB Ingestion

Ingesting raw json data extracted from coffeereviews.com to DuckDB

In [1]:
import json
import duckdb

Check the unique keys in the JSON file

In [11]:
path = "../data/raw"
with open(f"{path}/coffee.json", 'r') as file:
  data = json.load(file)

unique_keys = set()
for item in data:
  if isinstance(item, dict):
    unique_keys.update(item.keys())

list(unique_keys)

['Aroma',
 'Notes',
 'Roast Level',
 'rating',
 'Roaster Location',
 'Flavor',
 'Agtron',
 'Acidity',
 'Who Should Drink It',
 'bean',
 'Explore Similar Coffees',
 'With Milk',
 'Acidity/Structure',
 'Body',
 'Coffee Origin',
 'Aftertaste',
 'Bottom Line',
 'Review Date',
 'Est. Price',
 'Blind Assessment',
 'roaster']

Load into DuckDB @ `/data/warehouse/coffee.duckdb`

In [12]:
JSON_PATH = "../data/raw/coffee.json"
DB_PATH = "../data/warehouse/coffee.duckdb"

con = duckdb.connect(DB_PATH)

# Create the schema
# con.execute("CREATE SCHEMA IF NOT EXISTS coffee;")
con.execute("DROP TABLE IF EXISTS reviews;")

# Data ingestion
con.execute(f"""
  CREATE TABLE reviews AS
    SELECT
      TRY_CAST(data->>'$.rating' AS INTEGER) AS rating,
      (data->>'$.roaster') AS roaster,
      (data->>'$.bean') AS bean,
      (data->>'$.Roaster Location') AS location,
      (data->>'$.Coffee Origin') AS origin,
      (data->>'$.Roast Level') AS roast_level,
      (data->>'$.Agtron') AS agtron,
      (data['Est. Price']) AS est_price,
      strptime(data->>'$.Review Date', '%B %Y')::DATE AS review_date,
      TRY_CAST(data->>'$.Aroma' AS INTEGER) AS aroma,
      TRY_CAST(data->>'$.Body' AS INTEGER) AS body,
      TRY_CAST(data->>'$.Flavor' AS INTEGER) AS flavor,
      TRY_CAST(data->>'$.Aftertaste' AS INTEGER) AS aftertaste,
      TRY_CAST(data->>'$.With Milk' AS INTEGER) AS with_milk,
      TRY_CAST(data->>'$.Acidity/Structure' AS INTEGER) AS acid_structure,
      (data->>'$.Acidity') AS acidity,
      (data->>'$.Blind Assessment')::TEXT AS blind_assessment,
      (data->>'$.Notes')::TEXT AS notes,
      (data->>'$.Who Should Drink It')::TEXT AS who_should_drink,
      (data->>'$.Bottom Line')::TEXT AS bottom_line
    FROM
      read_json('{JSON_PATH}', format='auto') AS data;
""")

In [13]:
con.sql("SHOW ALL TABLES")

┌──────────┬─────────┬─────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────┐
│ database │ schema  │  name   │                                                                                                    column_names                                                                                                     │                                                                                   column_types                                                                                    │ temporary │
│ varchar  │ varchar │ varchar │                                                                                          

In [14]:
con.sql("SELECT * FROM coffee.reviews")

┌────────┬────────────────────────────┬─────────────────────────────────────────────────────────┬───────────────────────────────┬────────────────────────────────────────────────────┬──────────────┬─────────┬────────────────────┬─────────────┬───────┬───────┬────────┬────────────┬───────────┬────────────────┬─────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [10]:
con.close()

# Data Exploration

Exploring the retrieved data

In [2]:
DB_PATH = "../data/warehouse/coffee.duckdb"

con = duckdb.connect(DB_PATH)

In [18]:
# check for missing entries
con.sql("SELECT DISTINCT aroma FROM coffee.reviews;")

┌───────┐
│ aroma │
│ int32 │
├───────┤
│     2 │
│     4 │
│     9 │
│     7 │
│    10 │
│     3 │
│  NULL │
│     6 │
│     8 │
│     5 │
└───────┘

In [7]:
con.close()

# ChromaDB Check

Checking the entries in the ChromaDB.

In [22]:
import chromadb
from pprint import pprint

In [24]:
client = chromadb.PersistentClient(path="../data/chroma")
collection = client.get_collection(name="coffee")

In [26]:
results = collection.query(
  query_texts=["I like bold and aromatic coffees"],
  n_results=2
)
print(results['documents'][0][0])

The Seattle’s Best Blend is a bean from Seattle, Washington which originated in Indonesia, Central and South America.
    Roasted by Seattle's Best Coffee. 

    OVERALL RATING: 88

    Flavor Profile:
    Roast Level: Medium-Dark
    Agtron: 47/52
    Aroma: 7
    Body: 8
    Flavor: 8
    Aftertaste: 
    With Milk: 
    Acidity/ Structure: 
    Acidity: 6

    Blind Assessment
    This coffee gets better as it goes, probably owing to its heavy body. The dark tones of the aroma modulate with increasing power through the cup to a complex finish.

    Notes
    Coffee with a heavy body that get better as it goes.

    Who Should Drink It
    Classicists who brew with a French press; people who like a cup with power but low acidity; people who love coffee but still pour milk into it. Serve it to business meetings involving people from Boston and Seattle. Neither side may be completely happy, but they won't carp either. Should make an excellent espresso.


In [27]:
con.sql("SELECT * FROM coffee.reviews WHERE bean = 'Seattle’s Best Blend'")

┌────────┬───────────────────────┬──────────────────────┬─────────────────────┬──────────────────────────────────────┬─────────────┬─────────┬───────────┬─────────────┬───────┬───────┬────────┬────────────┬───────────┬────────────────┬─────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────┐
│ rating │        roaster        │         bean         │      location       │                origin                │ roast_level │ agtron  │ est_price │ review_date │ aroma │ body  │ flavor │ afte

In [2]:
%watermark

Last updated: 2025-06-18T19:03:45.452311+08:00

Python implementation: CPython
Python version       : 3.11.9
IPython version      : 8.31.0

Compiler    : MSC v.1938 64 bit (AMD64)
OS          : Windows
Release     : 10
Machine     : AMD64
Processor   : Intel64 Family 6 Model 183 Stepping 1, GenuineIntel
CPU cores   : 20
Architecture: 64bit

